# 07 Cron and Webhook Automations (OpenClaw, 2026)

## What This Lesson Is
Design robust automation triggers with authenticated webhooks and bounded retry semantics.

## Scientific Lens
- Concept: Automation reliability requires explicit trigger auth, backoff, and delivery strategy.
- Measure: Successful automation completion rate under retry policy.
- Validity Limit: Scheduler success may still depend on external systems beyond OpenClaw.


## How It Works
1. Model retry/backoff envelopes for scheduled jobs.
2. Validate webhook token requirements deterministically.
3. Run live OpenClaw prompt to produce automation failure triage plan.


In [ ]:
import os
from openai import OpenAI

OPENCLAW_BASE_URL = os.getenv("OPENCLAW_BASE_URL", "http://127.0.0.1:18789").rstrip("/")
OPENCLAW_TOKEN = os.getenv("OPENCLAW_GATEWAY_TOKEN") or os.getenv("OPENAI_API_KEY") or ""
OPENCLAW_TOKEN_SOURCE = (
    "OPENCLAW_GATEWAY_TOKEN" if os.getenv("OPENCLAW_GATEWAY_TOKEN")
    else ("OPENAI_API_KEY" if os.getenv("OPENAI_API_KEY") else "<missing>")
)
OPENCLAW_AGENT_ID = os.getenv("OPENCLAW_AGENT_ID", "main")

print("OPENCLAW_BASE_URL:", OPENCLAW_BASE_URL)
print("OPENCLAW_TOKEN source:", OPENCLAW_TOKEN_SOURCE)
print("OPENCLAW_AGENT_ID:", OPENCLAW_AGENT_ID)


def build_client() -> OpenAI:
    # OpenClaw exposes an OpenAI-compatible Chat Completions endpoint at /v1/chat/completions.
    # Docs: https://docs.openclaw.ai/gateway/openai-http-api
    return OpenAI(base_url=f"{OPENCLAW_BASE_URL}/v1", api_key=OPENCLAW_TOKEN or "local-dev-token")


def ask_openclaw(prompt: str, user: str = "lesson-user", temperature: float = 0.2) -> str:
    client = build_client()
    resp = client.chat.completions.create(
        model="openclaw",
        messages=[{"role": "user", "content": prompt}],
        user=user,
        temperature=temperature,
        extra_headers={"x-openclaw-agent-id": OPENCLAW_AGENT_ID},
    )
    return resp.choices[0].message.content or ""


## Code Walkthrough
- `Deterministic Demo` defines and validates the decision logic.
- `Live Demo` executes a real OpenClaw agent call through the OpenAI-compatible gateway API.


In [ ]:
# Deterministic Demo
jobs = [
    {"id":"A","retries":0,"delivery":"none"},
    {"id":"B","retries":2,"delivery":"announce"},
]
backoff = [30, 60, 300, 900, 3600]
plan = {j["id"]: backoff[:j["retries"]] for j in jobs}
assert plan["A"] == []
assert plan["B"] == [30, 60]


In [ ]:
# Live Demo
try:
    q = "Draft an incident triage flow for failed OpenClaw cron jobs with webhook delivery."
    print(ask_openclaw(q, user="cron-webhook-ops"))
except Exception as exc:
    print(f"Live demo call failed: {exc}")
    print("Set OPENCLAW_GATEWAY_TOKEN in .env (or export OPENAI_API_KEY) and rerun.")


## Applied Labs
1. Add exponential backoff cap and compare with fixed delay policy.
2. Add webhook replay protection constraints to deterministic model.
3. Define SLOs for automation success and lateness.

## Validation Checklist
- Retry policy is explicit and bounded.
- Webhook auth is treated as required control, not optional.
- Live call produces automation-specific response strategy.

## Further Reading
- OpenClaw cron docs: https://docs.openclaw.ai/automation/cron-jobs
- OpenClaw webhook docs: https://docs.openclaw.ai/automation/webhook
